# Named-entity recognition: discriminative and generative formulations

A pretrained BERT token classifier is evaluated on 64 WikiANN validation examples. An optional Gemma branch uses the same examples when `RUN_GENERATIVE_NER=True`. Predictions are mapped to word-level BIO tags and scored with exact entity-level precision, recall and F1.

In [2]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from seqeval.metrics import classification_report, f1_score

from nlp_portfolio.ner import (pipeline_entities_to_dict, text_entities_to_bio,
                               wikiann_gold_tags, align_word_labels)

MODEL_ID = "dslim/bert-base-NER"
dataset = load_dataset("wikiann", "en", split="validation[:64]")
label_names = dataset.features["ner_tags"].feature.names
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID)
recognizer = pipeline("token-classification", model=model, tokenizer=tokenizer,
                      aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)
print("examples:", len(dataset), "labels:", label_names)

examples: 64 labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


Generating train split: 100%|##########| 20000/20000 [00:00<00:00, 1271136.03 examples/s]


## Subword-label alignment

Special tokens receive `-100` so cross-entropy ignores them. When a word with `B-X` is split, subsequent pieces become `I-X`; otherwise a single entity would be counted as several beginnings.

In [4]:
row = dataset[0]
encoded = tokenizer(row["tokens"], is_split_into_words=True)
label_to_id = {name: index for index, name in enumerate(label_names)}
aligned = align_word_labels(encoded.word_ids(), row["ner_tags"], label_names, label_to_id)
pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(encoded["input_ids"]),
    "word_id": encoded.word_ids(),
    "training_label": aligned,
}).head(20)

,token,word_id,training_label
0,[CLS],NaN,-100
1,Sioux,0.0,3
2,Falls,1.0,4
3,Arena,2.0,4
4,(,3.0,0
5,Sioux,4.0,5
6,Falls,5.0,6
7,",",6.0,6
8,South,7.0,6
9,Dakota,8.0,6


In [5]:
gold, bert_predictions = [], []
for item in dataset:
    text = " ".join(item["tokens"])
    entities = pipeline_entities_to_dict(recognizer(text))
    gold.append(wikiann_gold_tags(item["ner_tags"], label_names))
    bert_predictions.append(text_entities_to_bio(item["tokens"], entities))

print("BERT entity-level F1:", round(f1_score(gold, bert_predictions), 4))
print(classification_report(gold, bert_predictions, digits=3))

BERT entity-level F1: 0.4238
              precision    recall  f1-score   support

    LOCATION      0.333     0.500     0.400        24
         ORG      0.588     0.333     0.426        30
      PERSON      0.625     0.357     0.455        28

   micro avg      0.464     0.390     0.424        82
   macro avg      0.516     0.397     0.427        82
weighted avg      0.526     0.390     0.428        82



## Prompted extraction with an instruction model

The generator returns JSON arrays of exact substrings. Invalid JSON becomes an empty prediction, so formatting failures reduce recall rather than being silently repaired with gold information.

In [7]:
RUN_GENERATIVE_NER = False
if RUN_GENERATIVE_NER:
    from transformers import AutoModelForCausalLM
    from nlp_portfolio.ner import predict_chat_entities

    GEMMA_ID = "unsloth/gemma-3-1B-it"
    chat_tokenizer = AutoTokenizer.from_pretrained(GEMMA_ID)
    chat_model = AutoModelForCausalLM.from_pretrained(
        GEMMA_ID, device_map="auto",
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    chat_predictions, raw_answers = [], []
    for item in dataset:
        entities, raw = predict_chat_entities(" ".join(item["tokens"]), chat_tokenizer, chat_model)
        chat_predictions.append(text_entities_to_bio(item["tokens"], entities))
        raw_answers.append(raw)
    print("Gemma entity-level F1:", round(f1_score(gold, chat_predictions), 4))

## Boundary-level error audit

Exact entity scoring penalizes type errors and boundary errors. The table below records the first disagreement in each sentence, which helps distinguish tokenization problems from missing entities.

In [9]:
errors = []
for item, expected, predicted in zip(dataset, gold, bert_predictions):
    mismatch = [i for i, (a, b) in enumerate(zip(expected, predicted)) if a != b]
    if mismatch:
        i = mismatch[0]
        errors.append({"text": " ".join(item["tokens"]), "token": item["tokens"][i],
                       "gold": expected[i], "prediction": predicted[i]})
pd.DataFrame(errors).head(12)

,text,token,gold,prediction
0,"Sioux Falls Arena ( Sioux Falls , South Dakota )",Sioux,B-ORG,B-LOCATION
1,"George Randolph Hearst , Jr .",George,B-PERSON,O
2,"Conch Key , Florida",",",I-LOCATION,O
3,Fairground]] '' by Simply Red '' Bad Romance '...,Simply,B-ORG,O
4,"The Game]] , Ice Cube , Dr. Dre",Ice,B-PERSON,O
5,Zina Garrison-Jackson '' ( Third Round ),Zina,B-ORG,O
6,' '' Bruno Ruffo '' ',Bruno,B-PERSON,O
7,Panamá ( 3 ),Panamá,B-LOCATION,B-ORG
8,**Monarch – Adolphe ( 1890–1905 ),–,O,B-PERSON
9,"The Edinburgh Rector is , uniquely , elected b...",Edinburgh,B-PERSON,B-LOCATION


The pretrained checkpoint and WikiANN do not share exactly the same annotation policy. The comparison therefore evaluates transfer under label mapping, not supervised fine-tuning. A full fine-tuning experiment should align labels before training and reserve a separate test split for the final score.

Recorded CPU run: BERT reached entity-level micro-F1 0.424 on 64 WikiANN validation examples (82 gold entities). Precision was 0.464 and recall was 0.390. This is a small cross-dataset transfer check; the pretrained model was not fine-tuned on WikiANN. The optional Gemma comparison was not run.